<a href="https://colab.research.google.com/github/jaehyeon0420/agent_tutorial/blob/master/rag_basic_example/rag_multi_turn%26query.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 이 파일은 LangChain과 같은 프레임워크 의존성 없이 텍스트 기반의 데이터를 활용하여 멀티턴 & 멀티쿼리를 구현하는 프로젝트입니다.

In [1]:
# OpenAI API, Vector DB, PDF 처리, 토큰 계산
!pip install -U openai chromadb pypdf tiktoken sentence-transformers

In [2]:
import os
import json
from typing import List, Dict

# OpenAI API 관련
from openai import OpenAI

# 벡터 DB 관련
import chromadb
from chromadb.utils import embedding_functions

# PDF 로딩 관련
from pypdf import PdfReader

from sentence_transformers import SentenceTransformer
import torch

from google.colab import userdata

import pandas as pd

from tqdm import tqdm

#### PDF의 모든 텍스트를 하나의 문자열로 병합합니다.

In [3]:
def load_pdf(file_path: str) -> str:
    reader = PdfReader(file_path)
    full_text = ""

    for page in reader.pages:
        # 페이지 텍스트 추출 (None 반환 대비)
        text = page.extract_text()
        if text:
            full_text += text + "\n"

    return full_text

pdf_text = load_pdf("대한민국 인공지능 행동계획.pdf")

#### 전체 문자열을 오버랩 기준으로 청킹하여 하나의 문자열 리스트를 생성합니다

In [4]:
def chunk_text(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:
    chunks = []

    # 시작 지점을 overlap만큼 이동하며 반복
    start = 0

    while start < len(text):
        # 종료 지점 계산
        end = start + chunk_size

        # 청킹
        chunk = text[start:end]
        chunks.append(chunk)

        # 다음 시작 지점 = 현재 종료 지점 - 오버랩
        start += (chunk_size - chunk_overlap)

        # 무한 루프 방지 (오버랩이 청크 사이즈보다 크거나 같을 경우)
        if chunk_size <= chunk_overlap:
            break

    return chunks

# 설정 값 예시
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

chunks = chunk_text(pdf_text, CHUNK_SIZE, CHUNK_OVERLAP)

In [5]:
chunks[0]

'대한민국 인공지능 행동계획\n2026. 2. 25\n국가인공지능전략위원회\n- 2 -\n≪들어가며≫인공지능(AI)은 과학적 발견의 방식을 근본적으로 바꾸고, 산업의 구조를 다시 짜며, 인간의 사고와 소통, 학습의 영역까지 재정의하고 있습니다. 이 거대한 기술혁명은 세계 질서를 재편하고, 경제와 안보의 규칙을 새로 만듭니다. 우리는 반도체·이동 통신·자동차 산업에서 그랬던 것처럼, 인공지능(AI)을 새로운 도약의 발판으로 삼아 온 국민이 혜택을 얻는 인공지능 기본사회를 이루고자 합니다. 능동적이고 자주적인 대응으로 이 변화를 우리의 것으로 만드는 것은 우리의 과제입니다. 「대한민국 인공지능 행동계획」은 이러한 비전을 실현하기 위해 마련되었습니다. 이 계획은 단순한 기술정책이 아니라, 인공지능을 새로운 국가 성장엔진으로 삼아 산업을 혁신하고, 국민의 삶을 개선하며, 인류 공동의 번영에 기여하고자 합니다. 행동계획은 세가지 정책축과 12대 전략분야로 구성되었습니다:먼저 첫 번째 축은 인공지능 혁신 생태계 조성입니다.인공지능 경쟁의 승패는 일차적으로 인프라 역량의 보유 규모와 활용 역량에 좌우됩니다. 대한민국은 데이터, 컴퓨팅, 반도체, 전력 등 인공지능 개발･활용의 기초가 되는 국가 인프라인 AI고속도로를 구축하고, 이를 바탕으로 인공지능 생태계 전반의 혁신에 나서야 합니다. 또한 초거대 인공지능 모델과 핵심기술을 선점하고, 세계 최고 수준의 인공지능 인재를 양성하여 글로벌 인공지능 기술 개발의 맨 앞에 설 수 있어야 합니다. 이를 위해 역량 있는 인공지능 연구자와 기업들이 규제에 가로막히지 않고 혁신에 도전할 수 있도록 법·제도·행정 전반의 구조를 재편해야 합니다. 이러한 측면에서 첫 번째 축은 다섯 가지 전략분야로 구성됩니다: ① AI고속도로 구축, ② 차세대 인공지능 기술 선점, ③ 인공지능 핵심인재 확보, ④ 독자 범용 인공지능 모델 확보, ⑤ 인공지능 규제혁신입니다. 이 5가지 분야는 인공지능 강국의 토대를 마련하고, 다음 세대의 국가 경쟁력을 결정짓는 

In [6]:
chunks[1]

' 핵심인재 확보, ④ 독자 범용 인공지능 모델 확보, ⑤ 인공지능 규제혁신입니다. 이 5가지 분야는 인공지능 강국의 토대를 마련하고, 다음 세대의 국가 경쟁력을 결정짓는 출발점이 될 것입니다.\n- 3 -\n두 번째 축은 범국가 인공지능 기반 대전환입니다.인공지능은 더 이상 특정 산업의 도구가 아니라, 국가 전체의 시스템을 바꾸는 원천입니다. 대한민국은 인공지능과 함께 산업, 공공, 지역, 국방, 문화 등 모든 국가영역의 혁신을 이룩해야 합니다. 인공지능은 제조·에너지·금융·바이오·의료 등 주력 산업의 효율성을 극대화하고, 행정과 복지, 교육 등 공공서비스의 품질을 획기적으로 개선할 수 있습니다. 또한 지역거점별 인공지능 허브 구축을 통해 수도권과 지방의 격차를 줄이고, 인공지능을 활용한 지능형 방위·안보 체계를 통해 첨단 국방 강국으로 도약해야 합니다. 아울러 우리가 가진 세계적인 장점인 K-문화와 K-콘텐츠산업에도 인공지능을 결합하여 창의성과 기술이 융합된 K-문화 르네상스를 열어가야 합니다. 이러한 측면에서 두 번째 축은 다섯 가지 전략분야로 구성됩니다: ⑥ 산업 인공지능 대전환, ⑦ 공공 인공지능 대전환, ⑧ 지역 인공지능 대전환, ⑨ 인공지능기반 문화강국, ⑩ 인공지능기반 국방강국 도약입니다. 이 5가지 전략 분야는 인공지능을 통해 국가 체계를 전면적으로 혁신하는 대한민국 인공지능 대전환 프로젝트입니다.마지막 축은 글로벌 인공지능 기본사회 기여입니다.인공지능 시대의 경쟁은 기술의 경쟁을 넘어 가치와 신뢰의 경쟁으로 진화하고 있습니다. 대한민국은 민주주의, 인권, 포용의 가치를 바탕으로 인공지능 기본사회를 실현하고, 국제사회가 신뢰할 수 있는 인공지능 거버넌스 모델을 제시해야 합니다. 인공지능이 국민 누구에게나 보편적 혜택으로 작동하고, 사회적 약자와 지역, 세대 간 격차를 줄이는 포용적 인공지능 복지 시스템을 구축하는 한편, 글로벌 인공지능 협력의 허브로서, 인공지능기술·데이터·인재 협력을 선도하고, 국제규범 논의와 공동 연구개발을 통해 세계 인공지

In [7]:
# 임베딩 모델 로드
model_name = "BAAI/bge-m3"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(model_name, device=device)

# ChromaDB 로컬 벡터 DB 설정
db_client = chromadb.PersistentClient(path="/DB")

# 컬렉션 생성
collection = db_client.get_or_create_collection(name="pdf_rag_collection")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [8]:
import uuid

def add_chunks_to_chroma(chunks: list[str], metadata_list: list[dict] = None):
    """
    청크 리스트를 임베딩하여 ChromaDB에 저장합니다.
    """
    # 임베딩 생성 (최대 2048개까지 한 번에 가능)
    embeddings = model.encode(
        chunks,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    # numpy 배열을 리스트로 변환
    embeddings = embeddings.tolist()

    # ChromaDB에 저장할 ID 생성 (중복 방지를 위해 UUID 사용)
    ids = [str(uuid.uuid4()) for _ in range(len(chunks))]

    # DB 적재
    # 만약 metadata_list가 없다면 기본값 설정
    if metadata_list is None:
        metadata_list = [{"source": "pdf_document"} for _ in range(len(chunks))]

    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=chunks,
        metadatas=metadata_list
    )
    print(f"성공적으로 {len(chunks)}개의 청크를 적재했습니다.")

add_chunks_to_chroma(chunks)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

성공적으로 205개의 청크를 적재했습니다.


#### 사용자의 쿼리를 이전 대화 맥락 기반으로 재생성하고, 의도별로 쿼리를 분리하는 역할을 담당하는 QueryRewriter 클래스입니다

- 이전 질문 : 오늘 점심 메뉴 추천해줘
- 이전 응답 : 봄동 비빔밥은 어떠신가요? 3월에는 봄동이 제철이라......~
- 현재 질문 : 1월은? 그리고 3월 여행지도 추천해줄래?
- **재생성된 쿼리** : ['1월의 점심에 먹을 제철 메뉴를 추천해줘.', '3월에 가볼만한 여행지 추천해줘']



In [9]:
import json
import os
from openai import OpenAI

class AdvancedQueryRewriter:
    def __init__(self, model_name: str = "gpt-4o", temperature: float = 0):
        self.client = OpenAI(api_key=userdata.get('OPEN_AI_KEY'))
        self.model_name = model_name
        self.temperature = temperature
        self.system_prompt = """당신은 사용자의 이전 대화와 현재 질문을 기반으로 검색 쿼리를 재생성하는 전문가입니다.

        # 지침
        1. 이전 대화 맥락과 현재 질문을 고려하여 더 정확한 검색 쿼리를 생성하세요.
        2. 현재 질문이 간략하거나 이전 대화의 맥락을 참조하는 경우, 이전 대화를 고려하여 완전한 쿼리를 구성하세요.
        3. 이전 대화가 없거나 현재 질문과 관련이 없는 경우 현재 질문만 사용하세요.
        4. 현재 질문에 여러 개의 의도가 포함되어 있다면, 의도별로 별도의 쿼리로 분리하세요.
        5. 각 쿼리는 독립적으로 벡터 DB에서 검색될 수 있도록 완전하고 명확해야 합니다.
        6. 마크다운 형식으로 작성하지 마세요.

        # 예시
        이전 대화 질문 : 오늘 점심 메뉴 추천해줘
        이전 대화 답변 : 봄동 비빔밥은 어떠신가요? 3월에는 봄동이 제철이라 인기가 많습니다
        현재 질문 : 1월은? 그리고 3월 여행지도 추천해줄래?
        재생성 쿼리 : {
          "queries": ["1월의 점심에 먹을 제철 메뉴를 추천해줘1", "3월에 가볼만한 여행지 추천해줘"]
        }

        # 출력 형식
        반드시 아래의 JSON 형식으로만 응답하세요:
        {
          "queries": ["재생성된 쿼리1", "재생성된 쿼리2", ...]
        }"""

    def rewrite_query(self, prev_query: str, prev_response: str, current_query: str) -> list[str]:
        """
        이전 대화(prev_query, prev_response)와 현재 질문을 분석하여 멀티 검색 쿼리를 생성
        """
        # 이전 대화가 없고 단순 질문인 경우의 최적화 (API 호출 절약)
        if not prev_query or not prev_query.strip():
            # 물음표가 하나뿐인 단순 문장이면 굳이 LLM을 쓰지 않고 바로 리스트로 반환
            if current_query.count("?") <= 1 and len(current_query) < 50:
                return [current_query.strip()]

        try:
            # 메시지 구성
            user_content = f"""# 입력 데이터
            이전 대화 질문: {prev_query}
            이전 대화 답변: {prev_response}
            현재 질문: {current_query}

            # 출력
            JSON으로 재생성된 쿼리 목록을 작성하세요."""

            # OpenAI API 호출
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[
                    {"role": "system", "content": self.system_prompt},
                    {"role": "user", "content": user_content}
                ],
                temperature=self.temperature,
                response_format={"type": "json_object"} # JSON 포맷 강제
            )

            # 결과 파싱
            content = response.choices[0].message.content
            response_json = json.loads(content)

            # 결과 검증 및 반환
            if "queries" in response_json and isinstance(response_json["queries"], list):
                return response_json["queries"]
            else:
                return [current_query]

        # 예외 발생 시 안정성 보장을 위해 현재 질문 그대로 반환
        except json.JSONDecodeError:
            print("JSON 파싱 오류가 발생했습니다.")
            return [current_query]
        except Exception as e:
            print(f"쿼리 재생성 중 예외 발생: {e}")
            return [current_query]

#### MultiQueryRetriever는 AdvancedQueryRewriter를 사용해 이전 대화 기반으로 멀티 쿼리를 생성하고
#### 각 쿼리에 대해 벡터 검색을 수행한 뒤 결과를 병합하는 검색기 입니다
#### 최종적으로 딕셔너리(검색 결과와 MetaData) 리스트 형태로 반환합니다.

In [10]:
import os
from openai import OpenAI

class MultiQueryRetriever:
    """
    AdvancedQueryRewriter를 사용하여 여러 쿼리로 검색하고
    ChromaDB 결과를 통합하는 클래스입니다.
    """

    def __init__(self, collection, query_rewriter):
        self.collection = collection  # ChromaDB 컬렉션 객체
        self.query_rewriter = query_rewriter
        self.client = OpenAI(api_key=userdata.get('OPEN_AI_KEY'))

        # 멀티턴 맥락 저장소
        self.prev_query = ""
        self.prev_response = ""

    def update_conversation(self, query: str, response: str):
        """이전 대화 맥락 업데이트"""
        self.prev_query = query
        self.prev_response = response

    def _get_embedding(self, text: str):
        """질문을 벡터로 변환 (Native OpenAI 호출)"""
        text = text.replace("\n", " ")
        return model.encode(
                    [text],
                    convert_to_numpy=True,
                    show_progress_bar=False
                )[0].tolist()

    def retrieve(self, query: str, num_docs: int = 5) -> list[dict]:
        """
        1. 쿼리 재작성 및 확장
        2. 개별 검색 수행
        3. 결과 병합 및 중복 제거
        """
        # AdvancedQueryRewriter를 이용해 멀티쿼리 생성
        rewritten_queries = self.query_rewriter.rewrite_query(
            self.prev_query,
            self.prev_response,
            query
        )

        print(f"[*] 원본 질문: {query}")
        print(f"[*] 생성된 검색 쿼리: {rewritten_queries}")

        all_results = []
        seen_contents = set()

        # 각 쿼리별로 검색 수행
        for r_query in rewritten_queries:
            # 쿼리 임베딩
            query_vector = self._get_embedding(r_query)

            # ChromaDB 검색
            search_results = self.collection.query(
                query_embeddings=[query_vector],
                n_results=num_docs
            )

            # 결과 파싱 및 중복 제거
            # ChromaDB 결과는 [['doc1', 'doc2']] 형태이므로 인덱싱 주의
            for i in range(len(search_results['documents'][0])):
                content = search_results['documents'][0][i]
                metadata = search_results['metadatas'][0][i] or {}

                # 중복 검사 및 결과 병합
                if content not in seen_contents:
                    seen_contents.add(content)
                    # 메타데이터에 어떤 쿼리로 검색되었는지 기록 (디버깅용)
                    metadata["source_query"] = r_query

                    all_results.append({
                        "page_content": content,
                        "metadata": metadata
                    })

        print(f"[+] 최종 {len(all_results)}개의 고유 문서 추출 완료.")
        return all_results

#### AdvancedConversationalRAG는 AdvancedQueryRewriter와 MultiQueryRetriever를 연결하여
#### RAG 시스템을 구현하기 위한 메인 컴포넌트입니다.
#### 쿼리 재작성 및 멀티쿼리 생성 -> 멀티 쿼리 검색 -> LLM 기반 답변 생성 -> 대화 기록 업데이트 -> 반환

In [11]:
import os
from openai import OpenAI

class AdvancedConversationalRAG:
    def __init__(self, collection, model_name="gpt-4o"):
        # 하위 컴포넌트 초기화
        self.query_rewriter = AdvancedQueryRewriter(model_name=model_name, temperature=0) # 쿼리 재생성
        self.retriever = MultiQueryRetriever( # 검색 및 결과 통합
            collection=collection,
            query_rewriter=self.query_rewriter
        )

        # OpenAI 클라이언트 및 설정
        self.client = OpenAI(api_key=userdata.get('OPEN_AI_KEY'))
        self.model_name = model_name
        self.temperature = 0

    def _generate_response(self, query: str, context: str) -> str:
        """최종 답변 생성을 위한 LLM 호출"""
        system_prompt = """당신은 사용자 질문에 대한 정보를 제공하는 도우미입니다.
        제공된 정보를 바탕으로 답변하되, 다음 지침을 엄수하세요:
        1. 질문에 여러 요소가 있다면 각각에 대해 명확히 답변하세요.
        2. 제공된 정보(Context)만을 사용하여 답변하세요. 정보가 없으면 "모른다"고 답하세요.
        3. 답변은 한국어로 작성하세요.
        4. 답변 시 근거가 되는 문서의 정보(예: 출처, 페이지 등)가 있다면 인용하세요."""

        user_content = f"""# 사용자 질문: {query}

        # 참고 정보(Context):
        {context}

        # 답변:"""

        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content}
            ],
            temperature=self.temperature
        )
        return response.choices[0].message.content.strip()

    def query(self, current_query: str) -> dict:
        # 관련 문서 검색 (내부적으로 멀티턴/멀티쿼리 처리됨)
        docs = self.retriever.retrieve(current_query)

        # 문서 내용을 컨텍스트 문자열로 변환 (메타데이터 포함)
        context_parts = []
        for i, doc in enumerate(docs):
            source = doc['metadata'].get('source', '알 수 없음')
            # 컨텍스트에 메타데이터를 포함시켜 LLM이 인용할 수 있게 함
            context_parts.append(f"[문서 {i+1} / 출처: {source}]\n{doc['page_content']}")

        context = "\n\n".join(context_parts)

        # 최종 응답 생성
        response = self._generate_response(current_query, context)

        # 대화 기록 업데이트 (맥락 유지를 위함)
        self.retriever.update_conversation(current_query, response)

        # 결과 반환
        return {
            "query": current_query,
            "result": response,
            "source_documents": docs
        }

In [12]:
conversational_rag = AdvancedConversationalRAG(collection)

In [13]:
# 예시 1: 첫 번째 질문 (이전 대화 없음)
query1 = "산업 AX에 대한 비전이나 미션이 뭐야?"
print("\n" + "="*50)
print(f"질문 1: {query1}")
result1 = conversational_rag.query(query1)
print(f"답변 1:\n{result1['result']}")


질문 1: 산업 AX에 대한 비전이나 미션이 뭐야?
[*] 원본 질문: 산업 AX에 대한 비전이나 미션이 뭐야?
[*] 생성된 검색 쿼리: ['산업 AX에 대한 비전이나 미션이 뭐야?']
[+] 최종 5개의 고유 문서 추출 완료.
답변 1:
산업 AX에 대한 비전과 미션은 "AI × 5극 3특 = 깨어난 DNA+, 살아난 지역, 희망의 대한민국"을 목표로 하고 있습니다. 이는 인공지능을 매개로 한 권역별 AX 전환을 촉진하고, 5극 3특을 유기적으로 연결하여 국가 균형성장을 견인하는 대한민국형 지역혁신 모델을 구현하는 것입니다. 이러한 비전은 지역별로 분산된 산업, 인재, 데이터를 유기적으로 연결하여 AI 서비스의 품질과 적용 범위를 넓히고, 지속 가능한 성장동력을 만들어내는 것을 목표로 합니다. 이를 통해 수도권 집중형 성장 구조에서 벗어나 전국이 유기적으로 연계된 균형 있는 국가 성장 모델을 완성하고자 합니다. (출처: 문서 4)


In [14]:
# 예시 2: 후속 질문
query2 = "그럼 지역 AX는?"
print("\n" + "="*50)
print(f"질문 : {query2}")
result1 = conversational_rag.query(query2)
print(f"답변 :\n{result1['result']}")


질문 : 그럼 지역 AX는?
[*] 원본 질문: 그럼 지역 AX는?
[*] 생성된 검색 쿼리: ['지역 AX에 대한 비전이나 미션이 뭐야?']
[+] 최종 5개의 고유 문서 추출 완료.
답변 :
지역 AX는 "AI × 5극 3특 = 깨어난 DNA+, 살아난 지역, 희망의 대한민국"이라는 비전과 미션을 가지고, 인공지능을 매개로 한 권역별 AX 전환을 촉진하는 것을 목표로 합니다. 이를 통해 5극 3특을 유기적으로 연결하여 국가 균형성장을 견인하는 대한민국형 지역혁신 모델을 구현하고자 합니다. AI 기반의 지역 AX 전환은 각 지역의 분권형 거버넌스 확립, 네트워크형 인프라 구축, 산업·사회 전반의 지능화 가속, 삶의 질과 정주 여건 개선을 통해 지역 고유의 경쟁력과 자립 생태계를 회복하도록 지원합니다. 또한, 지역 간 연결성이 강화되면 디지털·AI 인프라가 전국적 네트워크로 확장되어 전국 단위의 동시 성장이 가능해지는 효과도 기대됩니다. (출처: 문서 1)


In [15]:
# 예시 3: 주제 전환
query2 = "AI 기본사회에 대해서 설명해줘"
print("\n" + "="*50)
print(f"질문 : {query2}")
result1 = conversational_rag.query(query2)
print(f"답변 :\n{result1['result']}")


질문 : AI 기본사회에 대해서 설명해줘
[*] 원본 질문: AI 기본사회에 대해서 설명해줘
[*] 생성된 검색 쿼리: ['AI 기본사회에 대한 설명을 제공해줘']
[+] 최종 5개의 고유 문서 추출 완료.
답변 :
AI 기본사회는 인공지능 기술의 발전으로 인한 혜택을 모든 국민이 골고루 누리는 사회를 목표로 합니다. 이는 기술 발전이 포용적 사회로의 발전 동력이 되는 것을 의미하며, 정부, 국회, 기업, 시민사회 등 모든 사회 구성원이 함께 책임지고 구현해야 할 국가적 목표입니다. 이를 위해 '모두의 AI' 플랫폼을 구축하여 국민이 AI와 AI에 의한 변화를 쉽게 이해하고 체험할 수 있도록 하며, 일상에서 자연스럽게 AI를 활용할 수 있는 환경을 조성합니다. 

AI 기본사회 추진 계획은 노동, 복지, 교육, 금융, 문화, 안전, 환경 등 7대 핵심 영역의 구현 전략을 포함하며, 공공과 민간이 협력하는 구조를 통해 법과 제도, 조직 차원에서 안정적으로 뒷받침하여 지속 가능하도록 만들어야 합니다. 또한, AI 기본사회는 국민의 기본권 확대와 사회적 격차 해소, 사회안전망 강화를 목표로 하며, AI 민주주의, 기술윤리, 사회적 포용원칙이 통합적으로 고려되어야 합니다. 

이러한 계획은 과기정통부와 행안부 등 관계 부처가 협력하여 수립하며, 민간의 다양한 의견을 포괄할 수 있도록 민간의 참여가 보장된 거버넌스 기구를 구축하여 추진됩니다. (출처: 문서 1, 2, 3, 4)
